# Baking: self-contained artifacts from any recipe

chebax instances can be "baked" into dependency-free artifacts:

- `bake.jax_module` writes a pure-jax Python module (imports jax only,
  not chebax) defining the function and its tables.
- `bake.xsf_header` writes a standalone C++17 header (only `<cmath>`)
  in the style of scipy's xsf, ready for host or device code.

Both are emitted from the instance's own traced computation, so the
artifact cannot diverge from the runtime: the Python artifact is
bit-identical to it. Use baking to hand a single function to a project
that should not grow a chebax dependency.

In [1]:
import importlib.util
import pathlib
import shutil
import subprocess
import tempfile

import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)

import chebax
from chebax import bake

workdir = pathlib.Path(tempfile.mkdtemp())

## A pure-jax module

In [2]:
inst = chebax.besselk(1.5)
mod_path = workdir / "besselk_baked.py"
name = bake.jax_module(inst, mod_path)
src = mod_path.read_text()
assert "import chebax" not in src
print(f"emitted {name}: {len(src.splitlines())} lines, no chebax import")
print()
print("\n".join(src.splitlines()[:14]))
print("    ...")

emitted besselk: 76 lines, no chebax import

"""BesselK (v = 1.5), baked by chebax 0.1.0.dev0 from the
runtime jaxpr. Self-contained: imports jax only. Do not edit.

Constants are python floats (weak typed), so the function follows the
dtype of x; plain jax AD gives the derivative."""

import jax
import jax.numpy as jnp

C_LTIL = (
    -1.538226699011144, -1.3822231619382443, -1.2000157852950335, -0.945950454526451,
    -0.6744283737836148, -0.4321110993387486, -0.24622691090607188, -0.12267015848923171,
    -0.05187371776464263, -0.017595323785010074, -0.004212034522497942, -0.000481786356359825,
    -6.1918503493327e-05, -0.0002348192116664268, -0.0002600788153626738, -0.00014900938010951387,
    ...


The body is the traced computation, flattened to one assignment per
primitive, with each Chebyshev series folded back to a `_clenshaw` call:

In [3]:
lines = src.splitlines()
start = next(i for i, l in enumerate(lines) if l.startswith(f"def {name}"))
print("\n".join(lines[start:start + 12]))
print("    ...")

def besselk(x):
    x = jnp.asarray(x)
    t1 = (x) < (1e-06)
    t2 = (x) <= (8.0)
    t3 = jnp.where(t2, x, 8.0)
    t4 = jnp.where(t1, 1e-06, t3)
    t5 = jnp.log(t4)
    t6 = _clenshaw((2.0 * (t5) - -11.736069016284437) / 15.89495209964411, C_LTIL)
    t7 = jnp.exp(t6)
    t8 = (t4) / (2.0)
    t9 = jnp.power(t8, -1.5)
    t10 = (t7) * (t9)
    ...


Load it with chebax nowhere in sight and compare against the runtime.

In [4]:
spec = importlib.util.spec_from_file_location("baked", mod_path)
baked = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baked)

x = jnp.asarray(np.logspace(-4, 2, 200))
print("max |baked - runtime|:",
      float(jnp.max(jnp.abs(getattr(baked, name)(x) - inst(x)))))
g_baked = jax.vmap(jax.grad(getattr(baked, name)))(x)
g_run = jax.vmap(jax.grad(inst))(x)
print("max relative grad diff:",
      float(jnp.max(jnp.abs(g_baked - g_run) / jnp.abs(g_run))))

max |baked - runtime|: 0.0


max relative grad diff: 7.725062821195886e-15


## A standalone C++ header

In [5]:
hdr_path = workdir / "besselk_baked.h"
cpp_name = bake.xsf_header(inst, hdr_path)
hdr = hdr_path.read_text()
fn_start = hdr.index("XSF_HOST_DEVICE inline double " + cpp_name)
print(hdr[fn_start:fn_start + 700])
print("...")

XSF_HOST_DEVICE inline double besselk_v1p5(double x) {
    using namespace detail_besselk_v1p5;
    bool t1 = (x) < (1e-06);
    bool t2 = (x) <= (8.0);
    double t3 = ((t2) ? (x) : (8.0));
    double t4 = ((t1) ? (1e-06) : (t3));
    double t5 = std::log(t4);
    double t6 = detail::clenshaw(detail::C_LTIL, (2.0 * (t5) - -11.736069016284437) / 15.89495209964411);
    double t7 = std::exp(t6);
    double t8 = (t4) / (2.0);
    double t9 = std::pow(t8, -1.5);
    double t10 = (t7) * (t9);
    bool t11 = (x) <= (8.0);
    double t12 = ((t11) ? (8.0) : (x));
    double t13 = (8.0) / (t12);
    double t14 = detail::clenshaw(detail::C_LTAIL, (2.0 * (t13) - 1.0) / 1.0);
    double t15 = std::exp(
...


Compile it with g++ (if available) and compare against the runtime.
Small differences are expected here: libm and XLA round `exp` and `log`
chains differently by a few ulp.

In [6]:
if shutil.which("g++") is None:
    print("g++ not available, skipping the compile check")
else:
    main = workdir / "main.cpp"
    main.write_text(
        f'#include "besselk_baked.h"\n#include <cstdio>\n'
        f'int main() {{ double x; while (std::scanf("%lf", &x) == 1) '
        f'std::printf("%.17e\\n", chebax_baked::{cpp_name}(x)); return 0; }}\n')
    exe = workdir / "exe"
    subprocess.run(["g++", "-O2", "-std=c++17", "-o", str(exe), str(main)],
                   check=True, cwd=workdir)
    out = subprocess.run([str(exe)], input="\n".join(repr(float(v)) for v in x),
                         capture_output=True, text=True, check=True)
    got = np.array([float(t) for t in out.stdout.split()])
    ref = np.asarray(inst(x))
    rel = np.max(np.abs(got - ref) / np.abs(ref))
    print(f"C++ vs runtime, max relative difference: {rel:.2e}")

C++ vs runtime, max relative difference: 8.79e-15


## Any recipe bakes

The emitter is generic over recipes with a closed-form evaluation
(solver-based functions like the quantiles are not bakeable yet). Same
call, different families:

In [7]:
for label, instance in [
    ("besselj(2.5)", chebax.besselj(2.5)),
    ("bessely(1.5)", chebax.bessely(1.5)),
    ("besseli(2.5, scaled)", chebax.besseli(2.5, scaled=True)),
    ("betainc(2, 3)", chebax.betainc(2.0, 3.0)),
    ("spherical_jn(2)", chebax.spherical_jn(2)),
]:
    p = workdir / (label.split("(")[0] + ".py")
    n = bake.jax_module(instance, p)
    print(f"{label:22s} -> {p.name:15s} ({len(p.read_text().splitlines())} lines, "
          f"function {n})")

besselj(2.5)           -> besselj.py      (93 lines, function besselj)
bessely(1.5)           -> bessely.py      (147 lines, function bessely)
besseli(2.5, scaled)   -> besseli.py      (78 lines, function besseli)
betainc(2, 3)          -> betainc.py      (88 lines, function betainc)


spherical_jn(2)        -> spherical_jn.py (101 lines, function spherical)
